# Figure 3
## GO setup
### Author: Martin Loza
### Date: 25/12/10

After selecting the window for downstream analysis, we will select the list of genes to perform GO in DAVID or panther

In [36]:
# Change R language to English
Sys.setenv(LANGUAGE = "en")

# Init
suppressPackageStartupMessages({
    library(dplyr)
    library(stringr)
    library(ggplot2)
    library(patchwork)
    library(dplyr)
})

# Local variables 
seed = 777
date = "251210"

# Define colors for strand plots
red = "#E41A1C"
blue = "#377EB8"
# Define colors for gene types
green = "#4DAF4A"
purple = "#984EA3"
text_size = 18
width = 18.6
dot_size = 4
line_size = 1.5
dpi = 300

in_dir = "/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/Data/Annotated_ncRNA_PCG_pairs/"
out_dir = "/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/04_Figure_3/Results/"
ensembl_dir = "/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/Data/ENSEMBL/selected/"
# Local Functions



### Load and setup the data

In [26]:
# Load the setup transcripts data
# We have different species, so let's create a list to store the data
data_list = list()

# Search for the available files
files <- list.files(in_dir)

# Load the data for each species
for (file in files) {
    # Remove the underscore and everything after it to get the species names
    species_name <- str_replace(file, "_.*", "")
    data_list[[species_name]] <- read.table(file.path(in_dir, file), sep = "\t", header = TRUE, 
                                            stringsAsFactors = FALSE, quote = "", 
                                            comment.char = "", fill = TRUE, row.names = NULL)
}

head(data_list[["human"]], 3)

,chromosome,ncRNA_id,ncrna_tss,ncrna_gene_name,ncrna_strand,gene_biotype,pcg_id,pcg_gene_name,pcg_tss,dna_distance,strand_distance,Family,is_TF
,<chr>,<chr>,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<chr>,<lgl>
1,19,ENST00000221567,54532791,,1,lncRNA,ENST00000590333,BRSK1,55282072,749281,749281,NA,FALSE
2,19,ENST00000221567,54532791,,1,lncRNA,ENST00000635964,C19orf85,55464751,931960,931960,NA,FALSE
3,19,ENST00000221567,54532791,,1,lncRNA,ENST00000346968,CACNG6,53992878,-539913,-539913,NA,FALSE


In [27]:
# Let's place an order in the plots
ordered_species <- c("human", "mouse")
# Arrange the data_list according to the ordered_species
data_selected_list <- data_list[ordered_species]

In [28]:
unique(data_selected_list[["human"]]$chromosome)

[1] "19" "12" "14" "1"  "7"  "20" "2"  "11" "22" "Y"  "16" "17" "X"  "3"  "6" 
[16] "10" "21" "9"  "5"  "8"  "15" "4"  "18" "13"

In [29]:
# remove data_list to avoid confusion
rm(data_list)

Let's focus only in lncRNAs

In [30]:
# Select only observations related to lncRNA
for (species in names(data_selected_list)) {
    data_selected_list[[species]] <- data_selected_list[[species]] %>%
        filter(gene_biotype == "lncRNA")
}

In [31]:
head(data_selected_list[["human"]], 3)
table(data_selected_list[["human"]]$gene_biotype)

,chromosome,ncRNA_id,ncrna_tss,ncrna_gene_name,ncrna_strand,gene_biotype,pcg_id,pcg_gene_name,pcg_tss,dna_distance,strand_distance,Family,is_TF
,<chr>,<chr>,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<chr>,<lgl>
1,19,ENST00000221567,54532791,,1,lncRNA,ENST00000590333,BRSK1,55282072,749281,749281,NA,FALSE
2,19,ENST00000221567,54532791,,1,lncRNA,ENST00000635964,C19orf85,55464751,931960,931960,NA,FALSE
3,19,ENST00000221567,54532791,,1,lncRNA,ENST00000346968,CACNG6,53992878,-539913,-539913,NA,FALSE



 lncRNA 
4155667 

Load the ENSEMBL annotations to map the gene ids

In [32]:
# create a list to store ENSEMBL annotations 
ensembl_annotations_list <- list()

# Get the files within the ENSEMBL directory
ensembl_files <- list.files(ensembl_dir)

# Select the files related to the species in data_selected_list
for (species in names(data_selected_list)) {
    ensembl_file <- ensembl_files[str_detect(ensembl_files, species)]
    ensembl_data <- read.table(file.path(ensembl_dir, ensembl_file), sep = "\t", header = TRUE, 
                               stringsAsFactors = FALSE, quote = "", 
                               comment.char = "", fill = TRUE, row.names = NULL)
    ensembl_annotations_list[[species]] <- ensembl_data
}

head(ensembl_annotations_list[["human"]], 3)


,row.names,Chromosome.scaffold.name,Gene.start..bp.,Gene.end..bp.,Strand,Gene.stable.ID,Transcript.stable.ID,Transcript.start..bp.,Transcript.end..bp.,Transcript.type,Gene.type,Gene.name,Gene.description,TSS,is_pcg,is_ncrna
,<chr>,<chr>,<int>,<int>,<int>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<int>,<lgl>,<lgl>
1,1,1,11121,24894,1,ENSG00000290825,ENST00000456328,11850,14416,lncRNA,lncRNA,DDX11L16,DEAD/H-box helicase 11 like 16 (pseudogene) [Source:NCBI gene (formerly Entrezgene);Acc:727856],11850,FALSE,TRUE
2,2,1,11121,24894,1,ENSG00000290825,ENST00000832823,14404,24894,lncRNA,lncRNA,DDX11L16,DEAD/H-box helicase 11 like 16 (pseudogene) [Source:NCBI gene (formerly Entrezgene);Acc:727856],14404,FALSE,TRUE
3,3,1,11121,24894,1,ENSG00000290825,ENST00000832824,11121,14413,lncRNA,lncRNA,DDX11L16,DEAD/H-box helicase 11 like 16 (pseudogene) [Source:NCBI gene (formerly Entrezgene);Acc:727856],11121,FALSE,TRUE


In [41]:
# Transfer the ensembl ids to the selected data
for (species in names(data_selected_list)) {
    data_selected_list[[species]] <- data_selected_list[[species]] %>%
        left_join(ensembl_annotations_list[[species]] %>%
                      select(Gene.stable.ID, Transcript.stable.ID) %>%
                      dplyr::rename(pcg_gene_ensembl_id = Gene.stable.ID,
                                           pcg_id = Transcript.stable.ID), 
                    by = c("pcg_id" = "pcg_id"))
}

### Selection of genes for GO analysis

Let's select the genes within the selected threshold for GO in DAIVD or Panther

In [42]:
# Add absolute distance column
for (species in names(data_selected_list)) {
    data_selected_list[[species]] <- data_selected_list[[species]] %>%
        mutate(abs_strand_distance = abs(strand_distance))
}

In [43]:
# Define the window sizes for downstream analysis
window_sizes <- 10000  # 10 kb

# Select only pairs within the defined window sizes
for (species in names(data_selected_list)) {
    data_selected_list[[species]] <- data_selected_list[[species]] %>%
        filter(abs_strand_distance <= window_sizes)
}

cat("Number of lncRNA-PCG pairs within", window_sizes, "bp window in human:\n")
nrow(data_selected_list[["human"]])
cat("Number of lncRNA-PCG pairs within", window_sizes, "bp window in mouse:\n")
nrow(data_selected_list[["mouse"]])

Number of lncRNA-PCG pairs within 10000 bp window in human:


[1] 105579

Number of lncRNA-PCG pairs within 10000 bp window in mouse:


[1] 52957

Now, let's select only unique gene names. In this case we don't have the gene ids, only the transcript Ids so we need to map the gene ids

In [45]:
head(data_selected_list[["human"]], 3)

,chromosome,ncRNA_id,ncrna_tss,ncrna_gene_name,ncrna_strand,gene_biotype,pcg_id,pcg_gene_name,pcg_tss,dna_distance,strand_distance,Family,is_TF,pcg_gene_ensembl_id,abs_strand_distance
,<chr>,<chr>,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<chr>,<lgl>,<chr>,<int>
1,20,ENST00000244070,58309454,,-1,lncRNA,ENST00000870895,RAB22A,58309706,252,-252,NA,FALSE,ENSG00000124209,252
2,11,ENST00000244906,61757671,MYRF-AS1,-1,lncRNA,ENST00000265460,MYRF,61755389,-2282,2282,NDT80_PhoG,TRUE,ENSG00000124920,2282
3,22,ENST00000248980,29478136,RFPL1S,-1,lncRNA,ENST00000310624,NEFH,29480218,2082,-2082,NA,FALSE,ENSG00000100285,2082


In [46]:
# Select unique pcg gene ensembl id for GO analysis
pcg_genes_list <- list()
for (species in names(data_selected_list)) {
    pcg_genes_list[[species]] <- unique(data_selected_list[[species]]$pcg_gene_ensembl_id)
}

cat("Number of unique PCG gene IDs for GO analysis in human:\n")
length(pcg_genes_list[["human"]])
cat("Number of unique PCG gene IDs for GO analysis in mouse:\n")
length(pcg_genes_list[["mouse"]])

Number of unique PCG gene IDs for GO analysis in human:


[1] 10082

Number of unique PCG gene IDs for GO analysis in mouse:


[1] 9848

In [47]:
# Save the selected data for downstream analysis
for (species in names(data_selected_list)) {
    output_file <- file.path(out_dir, paste0(species, "_PCG_list_within_", window_sizes, "bp_", date, ".tsv"))
    write.table(pcg_genes_list[[species]], file = output_file, sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE)
}